In [ ]:
import random

import numpy as np
import pandas as pd
import altair as alt

import helpers as h

from statsmodels.tsa.stattools import adfuller

%load_ext autoreload
%autoreload 2

In [ ]:
# Set random seeds
seed = 4
random.seed(seed)
np.random.seed(seed)

# Default altair chart properties
properties = {
    "width": 1000,
    "height": 250,
}

config = {"padding": 30}

## Random Walk

In [ ]:
n = 100
step = 2
intercept = 10

h.plot_timeseries(
    h.generate_random_walk(n, step, intercept),
    properties=properties,
    config=config,
    title="Random Walk"
)

In [ ]:
n = 100
intercept = -15
slope = 0.25

seasonality = [29, 30, 25, 27, 31, 43, 37]
seasonality_scale = 1, 1.5

# Generate a time series with a linear trend and seasonality.
raw = h.generate_linear(n, slope, intercept)
raw = h.add_seasonality(raw, seasonality, seasonality_scale)
raw = h.add_noise(raw, sd=1.2)

h.plot_timeseries(
    raw,
    properties={"width": 600, "height": 400},
    config=config,
)

## Auto Correlation

In [ ]:
# Generate autoregression series.
s = pd.Series([10, 11, 12, 13, 14, 15, 17, 18])
lag = s.shift(1)
s.corr(lag)

In [ ]:
s.autocorr(1)

In [ ]:
# Determine regression coefficient.
coef = lag.cov(s) / lag.var()
coef

In [ ]:
# Or via correlation and standard deviations.
coef = lag.corr(s) * (s[1:].std() / lag.std())
coef

In [ ]:
n = 5
intercept = 1

df = pd.DataFrame({"Time": range(n)})
for coef in (1.5, 1, 0.5, -0.5):
    df = df.assign(**{f"Coefficient {coef}": [intercept * coef ** _ for _ in range(n)]})
df = df.melt(id_vars="Time", var_name="Series", value_name="Value")


(
    alt
    .Chart(df)
    .mark_line(point=True)
    .encode(x="Time", y="Value", color="Series")
    .configure(padding=30)
)

## Differencing

In [ ]:
n = 28
slope = 1
step = 1
intercept = 6

lin = h.generate_linear(n, slope=slope, intercept=intercept)
lin_chart = h.plot_timeseries({"y": lin, "diff(1)": lin.diff(1)})

# walk = h.generate_random_walk(n, step, intercept)
slin = h.generate_sigmoid(n, exponent=2.1, scale=1)
slin = h.add_seasonality(lin, pattern=[1, 2, 1.3, 1, 1.5, 3, 4])

walk_chart = h.plot_timeseries({"y": slin, "diff(1)": slin.diff(1)})


(lin_chart & walk_chart).resolve_scale(y="shared").configure(padding=50)

In [ ]:
n = 100
intercept = 10
slope = 1

seasonality = [29, 30, 25, 27, 31, 43, 37]
seasonality_scale = 1.5, 7

# Generate a time series with a linear trend and seasonality.
raw = h.generate_linear(n, slope, intercept)
raw = h.add_seasonality(raw, seasonality, seasonality_scale)
raw = h.add_noise(raw, sd=4)

h.plot_timeseries(
    raw,
    properties={"width": 600, "height": 400},
    config=config,
    title="Non-stationairy Series"
)

In [ ]:
# Plot stationary vs non-stationary.
n = 50
stationary = h.generate_autoregression(n, 0.9, 0)
trend = h.add_linear(stationary, slope=0.2)

(
    (h.plot_timeseries(trend, title="Series with trend") & h.plot_timeseries(stationary, title="Stationary series"))
    .resolve_scale(y="shared")
    .configure(padding=30)
)


In [ ]:
# Test for trend for stationarity.
result = adfuller(trend)
print(f"ADF statistic: {result[0]:5.2f}")
print(f"p-value:       {result[1]:5.2f}")

In [ ]:
# Test for stationary for stationarity.
result = adfuller(stationary)
print(f"ADF statistic: {result[0]:5.2f}")
print(f"p-value:       {result[1]:5.2f}")

## Moving Average

In [ ]:
# Set random seed
np.random.seed(7)

n = 50
walk = h.generate_random_walk(n, step=2, intercept=10).astype(float)

h.plot_timeseries(
    {"Walk": walk, "MovingAverage": walk.rolling(3).mean()},
    colors={"Walk": "#ff7f0e", "MovingAverage": "#1f77b4"},
    properties=properties,
    
).configure(padding=30)

In [ ]:
walk[12] = 0

walk_chart = h.plot_timeseries(
    {"Walk": walk, "MovingAverage": walk.rolling(3).mean()},
    colors={"Walk": "#ff7f0e", "MovingAverage": "#1f77b4", "Delta": "#ff7f0e"},
    properties=properties,
)

delta_chart = h.plot_timeseries(
    {"Delta": walk - walk.rolling(3).mean()},
    mark="bar",
    properties = {
        "width": properties["width"],
        "height": properties["height"] // 3,
    }
)

(walk_chart & delta_chart).configure(padding=30)

In [ ]:
n = 100
seasonality = [29, 30, 25, 27, 31, 43, 37]

# Generate a time series with a linear trend and seasonality.
raw = h.generate_sigmoid(n, exponent=1.5, scale=120)
raw = h.add_seasonality(raw, seasonality, 3)
raw = h.add_noise(raw, sd=3)


walk_chart = h.plot_timeseries(
    {"Series": raw, "MovingAverage": raw.rolling(7).mean()},
    colors={"Series": "#1f77b4", "MovingAverage": "#ff7f0e", "Delta": "#ff7f0e"},
    properties=properties,
)

delta_chart = h.plot_timeseries(
    {"Delta": raw - raw.rolling(7).mean()},
    # mark="bar",
    properties = {
        "width": properties["width"],
        "height": properties["height"] // 3,
    }
)

(walk_chart & delta_chart).configure(padding=30)

In [ ]:
# Seasonal Trend Decomposition using statsmodels.
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

# Decompose the time series using STL.
stl = STL(raw, period=7)
decomp = stl.fit()

# Access the components.
decomp.trend
decomp.seasonal
decomp.resid

# Plot the decomposition.
plt.rcParams["figure.figsize"] = (3, 8)
decomp.plot()
None


In [ ]:
# Seasonal Trend Decomposition using sktime
from sktime.transformations.series.detrend import STLTransformer

transformer = STLTransformer(sp=7)
transformer.fit(raw)

In [ ]:
# Plot STL decomposition using Altair.
import altair as alt

series = {
    "Raw": raw,
    "Trend": transformer.trend_,
    "Seasonality": transformer.seasonal_,
    "Residual": transformer.resid_,
}

charts = None

for name, data in series.items():
    df = pd.DataFrame({"Time": data.index, f"{name}": data})

    chart = (
        alt.Chart(df)
        .mark_line()
        .encode(
            x="Time:Q",
            y=alt.Y(f"{name}:Q"),
        )
    )

    charts = chart if charts is None else charts & chart


charts.configure(padding=50)